# Query-to-SubData Selection trên Google Colab

Notebook chỉ thực hiện **Data Discovery**: tạo biểu diễn nhẹ, route query theo corpus → document → page và lưu SubData manifest vào Google Drive.

> Notebook **không full parsing, không chunking và không full embedding** dữ liệu đã chọn.

## 1. Clone repository và cài thư viện

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/ManhTanTran/data-discovery.git"
REPO_DIR = Path("/content/data-discovery")

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[ml]"],
    check=True,
)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Đã cài đặt Data Discovery thành công.")

## 2. Kết nối Google Drive

Nếu dữ liệu nằm trong **Shared with me**, hãy tạo shortcut của `vidore_v3_industrial` vào **My Drive**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Điền đường dẫn trực tiếp nếu notebook không tự tìm thấy.
DATA_DIR_OVERRIDE = ""
my_drive = Path("/content/drive/MyDrive")

if DATA_DIR_OVERRIDE:
    data_dir = Path(DATA_DIR_OVERRIDE)
else:
    candidates = [
        my_drive / "vidore_v3_industrial" / "pdfs",
        my_drive / "iSE_DE" / "vidore_v3" / "vidore_v3_industrial" / "pdfs",
    ]
    data_dir = next((path for path in candidates if path.exists()), None)
    if data_dir is None:
        matches = list(my_drive.glob("**/vidore_v3_industrial/pdfs"))
        data_dir = matches[0] if matches else None

if data_dir is None or not data_dir.exists():
    raise FileNotFoundError(
        "Không tìm thấy thư mục pdfs. Hãy tạo shortcut hoặc đặt DATA_DIR_OVERRIDE."
    )
print(f"Dữ liệu: {data_dir}")
print(f"Số PDF: {len(list(data_dir.glob('*.pdf')))}")

## 3. Cấu hình Data Discovery

In [ ]:
import torch
from src.data_discovery import DiscoveryConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LIGHT_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
config = DiscoveryConfig(
    top_k_corpora=1,
    top_k_documents=8,
    top_k_pages=12,
    late_interaction_top_k=3,
    selection_threshold=0.15,
    alpha=0.25,
    beta=0.60,
    gamma=0.15,
    exploration_rate=0.05,
    max_preview_chars=1200,
    max_preview_segments_per_document=64,
    create_pdf_thumbnails=False,
    ann_backend="faiss",
)
print(f"Thiết bị: {DEVICE} | Model nhẹ: {LIGHT_MODEL}")

## 4. Light Preparation và Light Index

Bước này chỉ đọc metadata và text preview giới hạn theo trang, không gọi LLM.

In [ ]:
from src.data_discovery import LightIndex, LightPreparer, SentenceTransformerEmbedder

# Có thể thêm nhiều corpus vào dictionary này.
corpus_roots = {"vidore_industrial": data_dir}
manifest = LightPreparer(config).prepare(corpus_roots)
light_embedder = SentenceTransformerEmbedder(
    LIGHT_MODEL, device=DEVICE, batch_size=64
)
light_index = LightIndex(manifest, light_embedder, ann_backend=config.ann_backend)

print(f"Corpus: {len(manifest.corpora)}")
print(f"Document: {len(manifest.documents)}")
print(f"Preview/page segment: {len(manifest.segments)}")
print("ANN backend:", light_index.backend_used)

## 5. Query và chọn Top-K SubData

In [ ]:
import pandas as pd
from IPython.display import display
from src.data_discovery import QueryRouter

QUERY = "Which page describes the system architecture and its main components?"
selection = QueryRouter(light_index, config).select(QUERY)

def selection_frame(items):
    return pd.DataFrame([
        {
            "item_id": item.item_id,
            "corpus_id": item.corpus_id,
            "document_id": item.document_id,
            "page_id": item.page_id,
            "score": round(item.score, 4),
            "lexical_score": round(item.lexical_score, 4),
            "semantic_score": round(item.semantic_score, 4),
            "metadata_score": round(item.metadata_score, 4),
            "exploration": item.exploration,
        } for item in items
    ])

print("Corpus được chọn:")
display(selection_frame(selection.corpora))
print("Document được chọn:")
display(selection_frame(selection.documents))
print("Page/segment được chọn:")
display(selection_frame(selection.pages))
print("Processing plan cho pipeline parsing tiếp theo:")
display(pd.DataFrame(selection.processing_plan))
print("Số item bị loại:", selection.eliminated)
print("Latency (ms):", selection.latency_ms)
print(f"Giảm parsing ước tính: {selection.cost.parsing_cost_reduction:.2%}")
print(f"Giảm embedding ước tính: {selection.cost.embedding_cost_reduction:.2%}")

## 6. Lưu SubData vào Google Drive

Kết quả được lưu tại `AXIOM_DE-RD/data/output/subdata/vidore_v3_industrial/<timestamp>/`. Hãy tạo shortcut của thư mục `AXIOM_DE-RD` từ **Shared with me** vào **My Drive**. `documents/` chứa nguyên file được chọn; `pages/` chứa từng page PDF được chọn. Notebook không OCR, chunking hoặc full embedding.

In [ ]:
from dataclasses import asdict
from datetime import datetime
import json
import shutil

document_map = manifest.document_map()
segment_map = manifest.segment_map()
page_to_segment = {}
for segment in manifest.segments:
    page_to_segment.setdefault(segment.page_id, segment)

selected_pages = []
for item in selection.pages:
    segment = page_to_segment.get(item.page_id)
    document = document_map.get(item.document_id) if item.document_id else None
    selected_pages.append({
        "corpus_id": item.corpus_id,
        "document_id": item.document_id,
        "page_id": item.page_id,
        "page_number_zero_based": segment.page_number if segment else None,
        "page_number_human": (segment.page_number + 1) if segment and segment.page_number is not None else None,
        "source_uri": document.uri if document else None,
        "title": document.title if document else None,
        "preview_text": segment.text if segment else None,
        "thumbnail_uri": segment.thumbnail_uri if segment else None,
        "score": item.score,
        "lexical_score": item.lexical_score,
        "semantic_score": item.semantic_score,
        "metadata_score": item.metadata_score,
        "exploration": item.exploration,
    })

subdata = {
    "query": QUERY,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "selected_corpora": [asdict(item) for item in selection.corpora],
    "selected_documents": [asdict(item) for item in selection.documents],
    "selected_pages": selected_pages,
    "processing_plan": selection.processing_plan,
    "latency_ms": selection.latency_ms,
    "eliminated": selection.eliminated,
    "estimated_cost": selection.to_dict()["cost"],
    "config": asdict(config),
}

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
# Shared with me không xuất hiện trực tiếp trong Colab; cần shortcut AXIOM_DE-RD ở My Drive.
OUTPUT_ROOT_OVERRIDE = ""
output_root = (
    Path(OUTPUT_ROOT_OVERRIDE)
    if OUTPUT_ROOT_OVERRIDE
    else my_drive / "AXIOM_DE-RD" / "data" / "output"
)
if not output_root.parent.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {output_root.parent}. Hãy tạo shortcut AXIOM_DE-RD vào My Drive "
        "hoặc đặt OUTPUT_ROOT_OVERRIDE."
    )
output_dir = output_root / "subdata" / "vidore_v3_industrial" / run_id
output_dir.mkdir(parents=True, exist_ok=True)

# Copy nguyên file của các document được chọn; không parse nội dung.
documents_dir = output_dir / "documents"
documents_dir.mkdir(parents=True, exist_ok=True)
copied_documents = []
selected_document_ids = {
    item.document_id for item in selection.documents if item.document_id
}
for document_id in sorted(selected_document_ids):
    document = document_map[document_id]
    source_path = Path(document.uri)
    target_path = documents_dir / f"{document_id}_{source_path.name}"
    if not source_path.exists():
        copied_documents.append({
            "document_id": document_id,
            "source_uri": str(source_path),
            "copied_uri": None,
            "status": "source_not_found",
        })
        continue
    shutil.copy2(source_path, target_path)
    copied_documents.append({
        "document_id": document_id,
        "source_uri": str(source_path),
        "copied_uri": str(target_path),
        "status": "copied",
    })
subdata["copied_documents"] = copied_documents

# Tách mỗi page PDF được chọn thành một file PDF riêng; không OCR hoặc parse text.
import fitz

pages_dir = output_dir / "pages"
pages_dir.mkdir(parents=True, exist_ok=True)
extracted_pages = []
for page_record in selected_pages:
    source_path = Path(page_record["source_uri"]) if page_record["source_uri"] else None
    page_number = page_record["page_number_zero_based"]
    status = "unsupported_or_missing"
    extracted_uri = None
    if (
        source_path is not None
        and source_path.exists()
        and source_path.suffix.lower() == ".pdf"
        and page_number is not None
    ):
        human_page = int(page_number) + 1
        target_path = pages_dir / (
            f"{page_record['document_id']}_page_{human_page:04d}.pdf"
        )
        with fitz.open(source_path) as source_pdf:
            if 0 <= int(page_number) < len(source_pdf):
                with fitz.open() as page_pdf:
                    page_pdf.insert_pdf(
                        source_pdf, from_page=int(page_number), to_page=int(page_number)
                    )
                    page_pdf.save(target_path)
                extracted_uri = str(target_path)
                status = "extracted"
            else:
                status = "page_out_of_range"
    page_record["extracted_page_uri"] = extracted_uri
    page_record["extraction_status"] = status
    extracted_pages.append({
        "document_id": page_record["document_id"],
        "page_id": page_record["page_id"],
        "page_number_zero_based": page_number,
        "extracted_page_uri": extracted_uri,
        "status": status,
    })
subdata["selected_pages"] = selected_pages
subdata["extracted_pages"] = extracted_pages
(output_dir / "subdata_manifest.json").write_text(
    json.dumps(subdata, ensure_ascii=False, indent=2), encoding="utf-8"
)
pd.DataFrame(selected_pages).to_csv(
    output_dir / "selected_pages.csv", index=False, encoding="utf-8-sig"
)
pd.DataFrame(selection.processing_plan).to_json(
    output_dir / "processing_plan.json", orient="records", force_ascii=False, indent=2
)

print(f"Đã lưu SubData tại: {output_dir}")
print(f"- documents/ ({sum(item['status'] == 'copied' for item in copied_documents)} files)")
print(f"- pages/ ({sum(item['status'] == 'extracted' for item in extracted_pages)} files)")
print("- subdata_manifest.json")
print("- selected_pages.csv")
print("- processing_plan.json")

## Đầu ra cho bước parsing sau này

Pipeline ingestion chỉ cần đọc `processing_plan.json`, mở `source_uri` và xử lý các `page_numbers` tương ứng. Kết thúc Data Discovery tại đây.